# Ariel Data Challenge 2023 — EDA

**Docs**: https://www.ariel-datachallenge.space/ML/documentation/data

---

## What is this dataset?

Simulated exoplanet transmission spectra for the ESA ARIEL mission (launch 2029).  
Generated with **TauREx 3** (atmospheric model) + **ArielRad** (instrument simulator).

**Task**: Given a noisy spectrum + stellar/planetary features -> predict atmospheric composition.

## Input vs Output

### INPUT (both train & test)
| File | Content | Shape |
|------|---------|-------|
| `SpectralData.hdf5` | 52-bin transmission spectrum per planet | (N, 52, 4) |
| `AuxillaryTable.csv` | 8 stellar/planetary features | (N, 8) |

Spectral channels: wavelength, transit depth, noise, bin width

### OUTPUT / TARGETS (train only)
| File | Content |
|------|---------|
| `Tracedata.hdf5` | Full posterior distributions — only ~16% have actual traces, rest are empty |
| `QuartilesTable.csv` | q1/q2/q3 for 7 params (all 41,423 planets) |
| `FM_Parameter_Table.csv` | Forward model true params (all 41,423 planets) |

### 7 Target Parameters
planet_radius, T, log_H2O, log_CO2, log_CO, log_CH4, log_NH3

### CRITICAL: Test data has NO atmosphere labels
TestData (685 planets) has only spectra + auxiliary features. No ground truth.

## Setup

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('dark_background')
sns.set_palette('viridis')

ROOT = Path('ariel-ml-dataset')
TRAIN = ROOT / 'TrainingData'
TEST = ROOT / 'TestData'
GT = TRAIN / 'Ground Truth Package'

## 1. Auxiliary Features

In [ ]:
aux_train = pd.read_csv(TRAIN / 'AuxillaryTable.csv')
feat_cols = [c for c in aux_train.columns if c != 'planet_ID']
print(f'Shape: {aux_train.shape}')
print(f'Features: {feat_cols}')
aux_train.head()

In [ ]:
aux_train.describe()

In [ ]:
print('Missing values:')
missing = aux_train.isnull().sum()
print(missing[missing > 0] if missing.any() else 'None')

In [ ]:
ncols = 3
nrows = (len(feat_cols) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
for i, (ax, col) in enumerate(zip(axes.flat, feat_cols)):
    ax.hist(aux_train[col].dropna(), bins=50, alpha=0.7, edgecolor='none')
    ax.set_title(col, fontsize=9)
for j in range(i + 1, len(axes.flat)):
    axes.flat[j].set_visible(False)
plt.suptitle('Auxiliary Features Distributions (Train)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(aux_train[feat_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Auxiliary Features Correlation')
plt.tight_layout()
plt.show()

## 2. Spectral Data

In [ ]:
def load_spectra(hdf5_path, planet_ids):
    """Load spectra into (N, 52, 4): [wavelength, spectrum, noise, width]"""
    channels = ['instrument_wlgrid', 'instrument_spectrum',
                'instrument_noise', 'instrument_width']
    with h5py.File(hdf5_path, 'r') as f:
        spectra = np.empty((len(planet_ids), 52, 4))
        for idx, pid in enumerate(planet_ids):
            grp = f[f'Planet_{pid}']
            for ch_idx, ch in enumerate(channels):
                spectra[idx, :, ch_idx] = grp[ch][:]
    return spectra

planet_ids = aux_train['planet_ID'].values
spectra = load_spectra(TRAIN / 'SpectralData.hdf5', planet_ids)
print(f'Spectra shape: {spectra.shape}')

In [ ]:
ch_names = ['wavelength (um)', 'transit depth', 'noise', 'bin width']
for i, name in enumerate(ch_names):
    v = spectra[:, :, i]
    print(f'{name:20s} | min: {v.min():.6f} | max: {v.max():.6f} | mean: {v.mean():.6f} | std: {v.std():.6f}')

In [ ]:
wl = spectra[0, :, 0]
print(f'Wavelength range: {wl.min():.2f} - {wl.max():.2f} um ({len(wl)} bins)')
print(f'Grid identical across all planets: {np.all(spectra[:, :, 0] == wl)}')

In [ ]:
np.random.seed(42)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, idx in zip(axes.flat, np.random.choice(len(spectra), 6, replace=False)):
    ax.errorbar(spectra[idx, :, 0], spectra[idx, :, 1], yerr=spectra[idx, :, 2],
                fmt='o', markersize=2, capsize=1, alpha=0.8)
    ax.set_xscale('log')
    ax.set_xlabel('Wavelength (um)')
    ax.set_ylabel('Transit depth')
    ax.set_title(f'{planet_ids[idx]}', fontsize=10)
plt.suptitle('Example Transmission Spectra', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
snr = np.mean(spectra[:, :, 1] / spectra[:, :, 2], axis=1)
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(snr, bins=80, alpha=0.7, edgecolor='none')
ax.axvline(np.median(snr), color='red', ls='--', label=f'Median: {np.median(snr):.1f}')
ax.set_xlabel('Mean SNR')
ax.set_ylabel('Count')
ax.set_title('Signal-to-Noise Ratio Distribution')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Mean spectrum envelope across all planets
mean_spec = spectra[:, :, 1].mean(axis=0)
std_spec = spectra[:, :, 1].std(axis=0)
wl = spectra[0, :, 0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(wl, mean_spec - std_spec, mean_spec + std_spec, alpha=0.3)
ax.plot(wl, mean_spec, lw=1.5)
ax.set_xscale('log')
ax.set_xlabel('Wavelength (um)')
ax.set_ylabel('Transit depth')
ax.set_title(f'Mean Spectrum +/- 1 std ({len(spectra)} planets)')
plt.tight_layout()
plt.show()

## 2b. Spectral Anatomy — Instrument Channels & Wavelength Regions

The 52 wavelength bins span **0.55–7.28 μm** across 6 ARIEL instrument channels:

| Channel | Bins | Wavelength (μm) | Count | Purpose |
|---------|------|-----------------|-------|---------|
| **VISPhot** | 0 | 0.55 | 1 | Photometry (visible) |
| **FGS1** | 1 | 0.70 | 1 | Fine guidance sensor |
| **FGS2** | 2 | 0.95 | 1 | Fine guidance sensor |
| **NIRSpec** | 3–8 | 1.16–1.88 | 6 | Near-IR spectrometer |
| **AIRS-Ch0** | 9–41 | 1.97–3.72 | 33 | Main spectrometer |
| **AIRS-Ch1** | 42–51 | 4.03–7.28 | 10 | Long-wave spectrometer |

Most spectral information comes from **AIRS-Ch0** (33 bins, highest resolution).

In [ ]:
# Raw data sample — one planet's 52 bins with channel labels
CHANNELS = {
    'VISPhot': (0, 1), 'FGS1': (1, 2), 'FGS2': (2, 3),
    'NIRSpec': (3, 9), 'AIRS-Ch0': (9, 42), 'AIRS-Ch1': (42, 52)
}

def bin_to_channel(idx):
    for name, (lo, hi) in CHANNELS.items():
        if lo <= idx < hi:
            return name
    return '?'

wl = spectra[0, :, 0]
sample = pd.DataFrame({
    'bin': range(52),
    'wavelength_um': wl,
    'transit_depth': spectra[0, :, 1],
    'noise': spectra[0, :, 2],
    'bin_width': spectra[0, :, 3],
    'channel': [bin_to_channel(i) for i in range(52)]
})
print(f'=== Raw data for {planet_ids[0]} ===')
print(f'Shape per planet: (52, 4) = 52 wavelength bins x [wavelength, transit_depth, noise, bin_width]\n')
sample.style.background_gradient(subset=['transit_depth'], cmap='viridis')

In [ ]:
# Single spectrum with instrument channel bands + wavelength regions
CHANNEL_COLORS = {
    'VISPhot': '#e74c3c', 'FGS1': '#e67e22', 'FGS2': '#f1c40f',
    'NIRSpec': '#2ecc71', 'AIRS-Ch0': '#3498db', 'AIRS-Ch1': '#9b59b6'
}
WL_REGIONS = [
    (0.4, 0.7, 'Visible'),
    (0.7, 2.5, 'Near-IR'),
    (2.5, 8.0, 'Mid-IR'),
]

idx = 0
fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                      gridspec_kw={'height_ratios': [3, 1]})

# Top: spectrum with channel background bands
for ch_name, (lo, hi) in CHANNELS.items():
    wl_lo = spectra[idx, lo, 0] * 0.95
    wl_hi = spectra[idx, min(hi, 51), 0] * 1.03 if hi < 52 else spectra[idx, 51, 0] * 1.03
    ax_top.axvspan(wl_lo, wl_hi, alpha=0.15, color=CHANNEL_COLORS[ch_name], label=ch_name)

ax_top.errorbar(spectra[idx, :, 0], spectra[idx, :, 1], yerr=spectra[idx, :, 2],
                fmt='o-', markersize=4, capsize=2, color='white', linewidth=0.8, elinewidth=0.6)

for ch_name, (lo, hi) in CHANNELS.items():
    mask = slice(lo, hi)
    ax_top.scatter(spectra[idx, mask, 0], spectra[idx, mask, 1],
                   s=30, color=CHANNEL_COLORS[ch_name], zorder=5, edgecolors='white', linewidths=0.3)

ax_top.set_ylabel('Transit Depth', fontsize=12)
ax_top.set_title(f'Transmission Spectrum — {planet_ids[idx]} — by Instrument Channel', fontsize=14)
ax_top.legend(loc='upper right', fontsize=9, ncol=3)

# Bottom: wavelength regions + bin width
for wl_lo, wl_hi, label in WL_REGIONS:
    ax_bot.axvspan(wl_lo, wl_hi, alpha=0.2, color='gray')
    ax_bot.text((wl_lo + wl_hi) / 2, spectra[idx, :, 3].max() * 0.85, label,
                ha='center', fontsize=10, style='italic', color='white')

ax_bot.bar(spectra[idx, :, 0], spectra[idx, :, 3], width=0.08, alpha=0.7,
           color=[CHANNEL_COLORS[bin_to_channel(i)] for i in range(52)])
ax_bot.set_xlabel('Wavelength (μm)', fontsize=12)
ax_bot.set_ylabel('Bin Width (R)', fontsize=12)
ax_bot.set_xlim(0.4, 8.0)

plt.tight_layout()
plt.show()

In [ ]:
# Multi-planet comparison: spectra grouped by dominant molecule
fm_local = pd.read_csv(GT / 'FM_Parameter_Table.csv')
mol_cols = {'H₂O': 'log_H2O', 'CO₂': 'log_CO2', 'CH₄': 'log_CH4', 'NH₃': 'log_NH3'}

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, (mol_label, mol_col) in zip(axes.flat, mol_cols.items()):
    top_idx = fm_local[mol_col].nlargest(1).index[0]
    pid = fm_local.loc[top_idx, 'planet_ID']
    spec_idx = np.where(planet_ids == pid)[0][0]

    for ch_name, (lo, hi) in CHANNELS.items():
        wl_lo = spectra[spec_idx, lo, 0] * 0.95
        wl_hi = spectra[spec_idx, min(hi, 51), 0] * 1.03 if hi < 52 else spectra[spec_idx, 51, 0] * 1.03
        ax.axvspan(wl_lo, wl_hi, alpha=0.12, color=CHANNEL_COLORS[ch_name])

    ax.errorbar(spectra[spec_idx, :, 0], spectra[spec_idx, :, 1], yerr=spectra[spec_idx, :, 2],
                fmt='o-', markersize=3, capsize=1.5, color='white', linewidth=0.7, elinewidth=0.5)

    for ch_name, (lo, hi) in CHANNELS.items():
        ax.scatter(spectra[spec_idx, lo:hi, 0], spectra[spec_idx, lo:hi, 1],
                   s=20, color=CHANNEL_COLORS[ch_name], zorder=5)

    abundance = fm_local.loc[top_idx, mol_col]
    ax.set_title(f'High {mol_label} — {pid} (log={abundance:.1f})', fontsize=11)
    ax.set_xlabel('Wavelength (μm)')
    ax.set_ylabel('Transit Depth')

plt.suptitle('Spectra by Dominant Molecule — Channel-Colored', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. Ground Truth — Quartiles

In [ ]:
quartiles = pd.read_csv(GT / 'QuartilesTable.csv')
nan_rows = quartiles.isnull().any(axis=1).sum()
valid_rows = len(quartiles) - nan_rows
print(f'Quartiles: {quartiles.shape}')
print(f'Valid: {valid_rows} / {len(quartiles)} ({valid_rows/len(quartiles)*100:.1f}%)')
print(f'NaN rows: {nan_rows} ({nan_rows/len(quartiles)*100:.1f}%)')
print(f'Columns: {quartiles.columns.tolist()}')
quartiles.head()

In [ ]:
targets = ['planet_radius', 'T', 'log_H2O', 'log_CO2', 'log_CO', 'log_CH4', 'log_NH3']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, t in zip(axes.flat[:7], targets):
    col = f'{t}_q2'
    if col in quartiles.columns:
        data = quartiles[col].dropna()
        ax.hist(data, bins=50, alpha=0.7, edgecolor='none')
        ax.set_title(f'{t} (median)', fontsize=10)
        ax.text(0.98, 0.95, f'n={len(data)}', transform=ax.transAxes, ha='right', va='top', fontsize=8)
axes.flat[-1].set_visible(False)
plt.suptitle('Target Distributions (q2 = median, valid only)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, t in zip(axes.flat[:7], targets):
    q1, q3 = f'{t}_q1', f'{t}_q3'
    if q1 in quartiles.columns:
        ax.hist(quartiles[q3] - quartiles[q1], bins=50, alpha=0.7, edgecolor='none')
        ax.set_title(f'{t} (q3-q1)', fontsize=10)
axes.flat[-1].set_visible(False)
plt.suptitle('Target Uncertainty (q3 - q1)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. FM Parameters (Forward Model True Inputs)

In [ ]:
fm = pd.read_csv(GT / 'FM_Parameter_Table.csv')
fm_cols = [c for c in fm.columns if c not in ['Unnamed: 0', 'planet_ID']]
print(f'Shape: {fm.shape}')
print(f'Params: {fm_cols}')
fm.head()

In [ ]:
ncols = 3
nrows = (len(fm_cols) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 3.5 * nrows))
for i, (ax, col) in enumerate(zip(axes.flat, fm_cols)):
    ax.hist(fm[col].dropna(), bins=50, alpha=0.7, edgecolor='none')
    ax.set_title(col, fontsize=9)
for j in range(i + 1, len(axes.flat)):
    axes.flat[j].set_visible(False)
plt.suptitle('FM Parameters (True Forward Model Inputs)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4b. FM Truth vs Retrieved Quartiles

In [ ]:
# FM truth vs retrieved quartile medians
merged = fm.merge(quartiles, on='planet_ID')
fm_names = ['planet_radius', 'planet_temp', 'log_H2O', 'log_CO2', 'log_CO', 'log_CH4', 'log_NH3']
q_names = ['planet_radius_q2', 'T_q2', 'log_H2O_q2', 'log_CO2_q2', 'log_CO_q2', 'log_CH4_q2', 'log_NH3_q2']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, fm_col, q_col in zip(axes.flat[:7], fm_names, q_names):
    valid = merged[q_col].notna()
    ax.scatter(merged.loc[valid, fm_col], merged.loc[valid, q_col], s=1, alpha=0.3)
    lims = [min(merged.loc[valid, fm_col].min(), merged.loc[valid, q_col].min()),
            max(merged.loc[valid, fm_col].max(), merged.loc[valid, q_col].max())]
    ax.plot(lims, lims, 'r--', lw=0.8)
    ax.set_xlabel(f'FM truth')
    ax.set_ylabel(f'Retrieved q2')
    ax.set_title(fm_col, fontsize=10)
axes.flat[-1].set_visible(False)
plt.suptitle('Forward Model Truth vs Retrieved Median', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Ground Truth — Posteriors (Tracedata)

In [ ]:
trace = h5py.File(GT / 'Tracedata.hdf5', 'r')
trace_keys = list(trace.keys())

# Filter to planets with actual posterior data (non-empty tracedata)
valid_trace_keys = [k for k in trace_keys if len(trace[k]['tracedata'].shape) == 2]
empty_count = len(trace_keys) - len(valid_trace_keys)

print(f'Total entries: {len(trace_keys)}')
print(f'With posteriors: {len(valid_trace_keys)} ({len(valid_trace_keys)/len(trace_keys)*100:.1f}%)')
print(f'Empty: {empty_count} ({empty_count/len(trace_keys)*100:.1f}%)')

k0 = valid_trace_keys[0]
print(f'\n--- {k0} ---')
print(f'Subkeys: {list(trace[k0].keys())}')
td0 = trace[k0]['tracedata'][:]
wt0 = trace[k0]['weights'][:]
print(f'tracedata shape: {td0.shape}')
print(f'weights shape: {wt0.shape}')

In [ ]:
trace_lens = np.array([trace[k]['tracedata'].shape[0] for k in valid_trace_keys])
print(f'min: {trace_lens.min()}, max: {trace_lens.max()}, median: {np.median(trace_lens):.0f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(trace_lens, bins=50, alpha=0.7, edgecolor='none')
ax.set_xlabel('Trace points')
ax.set_ylabel('Count')
ax.set_title(f'Posterior Trace Lengths ({len(valid_trace_keys)} planets with data)')
plt.tight_layout()
plt.show()

In [ ]:
param_names = ['planet_radius', 'T', 'log_H2O', 'log_CO2', 'log_CO', 'log_CH4', 'log_NH3']
td = trace[k0]['tracedata'][:]
wt = trace[k0]['weights'][:].flatten()
wt = wt / wt.sum()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, (ax, name) in enumerate(zip(axes.flat[:7], param_names)):
    if i < td.shape[1]:
        ax.hist(td[:, i], weights=wt, bins=50, alpha=0.7, edgecolor='none')
        ax.set_title(name, fontsize=10)
axes.flat[-1].set_visible(False)
plt.suptitle(f'Weighted Posterior — {k0}', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 6. Train vs Test

In [ ]:
aux_test = pd.read_csv(TEST / 'AuxillaryTable.csv')
print(f'Train: {aux_train.shape}, Test: {aux_test.shape}')
print(f'Test columns: {aux_test.columns.tolist()}')
aux_test.head()

In [ ]:
shared = [c for c in feat_cols if c in aux_test.columns]
nrows = (len(shared) + 2) // 3
fig, axes = plt.subplots(nrows, 3, figsize=(14, 3.5 * nrows))
for i, (ax, col) in enumerate(zip(axes.flat, shared)):
    ax.hist(aux_train[col].dropna(), bins=40, alpha=0.6, label='train', density=True)
    ax.hist(aux_test[col].dropna(), bins=40, alpha=0.6, label='test', density=True)
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=8)
for j in range(i + 1, len(axes.flat)):
    axes.flat[j].set_visible(False)
plt.suptitle('Train vs Test Feature Distributions', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Summary

| | Train | Test |
|-|-------|------|
| **Planets** | 41,423 | 685 |
| **Spectra** | `SpectralData.hdf5` (181MB) | `SpectralData.hdf5` (3MB) |
| **Metadata** | `AuxillaryTable.csv` (8 features) | `AuxillaryTable.csv` (8 features) |
| **Quartiles** | all 41,423 | **NONE** |
| **FM params** | all 41,423 | **NONE** |
| **Posteriors** | 6,766 / 41,423 (16.3%) — rest empty | **NONE** |
| **Spectra shape** | (41423, 52, 4) | (685, 52, 4) |

### Targets: 7 parameters
planet_radius, T, log_H2O, log_CO2, log_CO, log_CH4, log_NH3

### Key notes
- Quartiles and FM params available for ALL train planets
- Full posteriors (tracedata) only for ~16% of train planets
- Test: spectra + features only, no labels
- planet_ID format: string like "train1", "train2", ... / "test1", "test2", ...
- HDF5 keys: `Planet_{planet_ID}` (e.g. `Planet_train1`)
- Tracedata subkeys: lowercase `tracedata`, `weights`
- Scoring: 80% posterior (KS test) + 20% spectral (Huber loss), max 1000

In [ ]:
trace.close()